In [2]:
import requests

API_HOST = "domainr.p.rapidapi.com"
API_KEY = "d0c57c7d69mshd3740bdbe30a37dp1a35c8jsn81117c113ab0"

def check_domain_availability(domain):
    url = f"https://{API_HOST}/v2/status"
    headers = {
        "x-rapidapi-host": API_HOST,
        "x-rapidapi-key": API_KEY
    }
    params = {"domain": domain}

    resp = requests.get(url, headers=headers, params=params, timeout=10)
    resp.raise_for_status()
    return resp.json()

In [4]:
import string

ignored_tlds = {".bl", ".bq", ".eh"}

with open("data/two_letter_tlds.txt", "r", encoding="utf-8") as f:
    tlds = [line.strip().lower() for line in f if line.strip() and line.strip().lower() not in ignored_tlds]

base_names = list(string.ascii_lowercase) + list(string.digits)
domains = [f"{base}{tld}" for base in base_names for tld in tlds]

print(f"Total domains generated: {len(domains)}")

Total domains generated: 9072


In [6]:
import csv
import os

# Check which domains have already been processed
processed_domains = set()
csv_file = "data/domains.csv"

if os.path.exists(csv_file):
    with open(csv_file, "r", newline="") as f:
        reader = csv.reader(f)
        next(reader)  # Skip header
        for row in reader:
            if row:  # Make sure row is not empty
                processed_domains.add(row[0])  # domain is in first column

print(f"Already processed {len(processed_domains)} domains")

# Filter out already processed domains
remaining_domains = [d for d in domains if d not in processed_domains]
print(f"Remaining domains to process: {len(remaining_domains)}")

# Resume processing with append mode
with open(csv_file, "a", newline="") as f:
    writer = csv.writer(f)
    
    # Only write header if file is empty/new
    if len(processed_domains) == 0:
        writer.writerow(["domain", "zone", "status", "summary"])

    for i, domain in enumerate(remaining_domains):
        print(f"Processing {i+1}/{len(remaining_domains)}: {domain}")
        
        try:
            payload = check_domain_availability(domain)
            
            for entry in payload.get("status", []):
                writer.writerow([
                    entry.get("domain", ""),
                    entry.get("zone", ""),
                    entry.get("status", ""),
                    entry.get("summary", "")
                ])
            f.flush()
            
        except Exception as e:
            print(f"Error processing {domain}: {e}")
            continue

Already processed 6 domains
Remaining domains to process: 9066
Processing 1/9066: a.al
Processing 2/9066: a.am
Processing 3/9066: a.an
Processing 4/9066: a.ao
Processing 5/9066: a.aq
Processing 6/9066: a.ar
Processing 7/9066: a.as
Processing 8/9066: a.at
Processing 9/9066: a.au
Processing 10/9066: a.aw
Processing 11/9066: a.ax
Processing 12/9066: a.az
Processing 13/9066: a.ba
Processing 14/9066: a.bb
Processing 15/9066: a.bd
Processing 16/9066: a.be
Processing 17/9066: a.bf
Processing 18/9066: a.bg
Processing 19/9066: a.bh
Processing 20/9066: a.bi
Processing 21/9066: a.bj
Processing 22/9066: a.bm
Processing 23/9066: a.bn
Processing 24/9066: a.bo
Processing 25/9066: a.br
Processing 26/9066: a.bs
Processing 27/9066: a.bt
Processing 28/9066: a.bv
Processing 29/9066: a.bw
Processing 30/9066: a.by
Processing 31/9066: a.bz
Processing 32/9066: a.ca
Processing 33/9066: a.cc
Processing 34/9066: a.cd
Processing 35/9066: a.cf
Processing 36/9066: a.cg
Processing 37/9066: a.ch
Processing 38/9066: a

In [27]:
import csv, os

csv_file = "data/domains.csv"
rows = {}  # domain -> dict(zone,status,summary)

# Load existing rows (if any)
if os.path.exists(csv_file):
    with open(csv_file, "r", newline="") as f:
        r = csv.reader(f)
        header = next(r, None)
        for row in r:
            if not row: 
                continue
            dom = row[0].strip()
            rows[dom] = {
                "zone": row[1].strip() if len(row) > 1 else "",
                "status": row[2].strip().lower() if len(row) > 2 else "",
                "summary": row[3].strip() if len(row) > 3 else "",
            }

ERROR_STATUSES = {"", "error", "failed", "timeout", "unknown"}

# Figure out which to (re)process
to_fix = [d for d in domains if d not in rows or rows[d]["status"] in ERROR_STATUSES]
print(f"Retrying {len(to_fix)} domains (missing or previously errored)…")

# Re-check and update in-memory rows
for d in to_fix:
    try:
        payload = check_domain_availability(d) or {}
        entries = payload.get("status", [])
        # Use the first matching entry (fallbacks if API returns nothing)
        e = next((e for e in entries if e.get("domain") == d), entries[0] if entries else {})
        rows[d] = {
            "zone": e.get("zone", ""),
            "status": (e.get("status", "") or "").lower(),
            "summary": e.get("summary", ""),
        }
    except Exception as ex:
        rows[d] = {"zone": "", "status": "error", "summary": f"retry_failed: {ex}"}

# Write back a clean CSV (header + all domains)
os.makedirs(os.path.dirname(csv_file), exist_ok=True)
with open(csv_file, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["domain", "zone", "status", "summary"])
    for d in sorted(rows.keys()):
        r = rows[d]
        w.writerow([d, r["zone"], r["status"], r["summary"]])

print("Done.")


Retrying 9 domains (missing or previously errored)…
Done.


In [28]:
# Print any that still failed after retry
still_bad = [d for d, r in rows.items() if r["status"] in ERROR_STATUSES]
print(f"{len(still_bad)} domains still failed after retry.")

if still_bad:
    print("These domains still failed:", still_bad)
else:
    print("✅ All domains processed successfully")


9 domains still failed after retry.
These domains still failed: ['1.sr', '4.mt', '6.vu', 'd.dj', 'i.nr', 'o.mt', 'r.cg', 's.mt', 't.kp']
